In [3]:
# To create custom return type, we can create our own class and use it as return type in the function.
# This is to be done inside the entity module.

# First create a dataclass (which is a constant class) for defining configuration of data ingestion.

from dataclasses import dataclass
from pathlib import Path
from CNNclassifier import *

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL : str
    local_data_file : Path
    unzip_dir : Path

In [4]:
from CNNclassifier.utils.common import read_yaml, create_directories
from CNNclassifier.constants import *

In [ ]:
# This funciton will define the configurations for data ingestion before downloading and extracting data
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )
        return data_ingestion_config
    


In [8]:
# Here we will define class and its functions to download and extract data from the source URL
# This will be done in the data_ingestion module.

import urllib.request as request
import zipfile
from CNNclassifier.utils.common import get_size


class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config

    # Function to download the data from the source URL and save it to the local data file path
    def download_file(self):
        if os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(url = self.config.source_URL, filename =self.config.local_data_file)
            logger.info(f"{filename} downloaded with following info :\n{headers}")
        else:
            logger.info(f"{filename} already exists of size : {get_size(Path(self.config.local_data_file))}")

    # Function to extract data from the zipped folder stored by download_file()
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.Zipfile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e